In [6]:
# ============================================================
# Stationarity + VAR Pipeline (Save CSV Outputs) - FIXED
# Path: ./merged_var_input.csv
# Outputs saved to ./
# ============================================================

import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

DATA_PATH = "./merged_var_input.csv"
MAX_LAGS = 10
ADF_AUTOLAG = "AIC"
MIN_ADF_N = 20  # too-short guard

# ----------------------------
# 1) Load
# ----------------------------
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

num_cols = [c for c in df.columns if c != "Date"]

print("Shape:", df.shape)
print("Date range:", df["Date"].min(), "~", df["Date"].max())
print("Columns:", num_cols)

# ----------------------------
# 2) ADF helper
# ----------------------------
def run_adf(series, autolag="AIC"):
    s = series.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < MIN_ADF_N:
        return {"adf_stat": np.nan, "pvalue": np.nan, "nobs": len(s), "status": "too_short"}
    if s.max() == s.min():
        return {"adf_stat": np.nan, "pvalue": np.nan, "nobs": len(s), "status": "constant"}
    res = adfuller(s.values, autolag=autolag)
    return {"adf_stat": res[0], "pvalue": res[1], "nobs": res[3], "status": "ok"}

# ----------------------------
# 3) RAW ADF
# ----------------------------
raw_rows = []
for col in num_cols:
    r = run_adf(df[col], ADF_AUTOLAG)
    raw_rows.append({"Variable": col, **r})

raw_adf = pd.DataFrame(raw_rows).sort_values(["status", "pvalue"], na_position="last")
raw_adf.to_csv("./adf_raw.csv", index=False)
print("Saved: ./adf_raw.csv")

# ----------------------------
# 4) Automatic Transform (logdiff if strictly positive else pct_change)
# ----------------------------
transformed = pd.DataFrame({"Date": df["Date"]})
transform_type = {}

for col in num_cols:
    s = df[col].astype(float)
    s_nonan = s.dropna()

    if len(s_nonan) > 0 and (s_nonan > 0).all():
        transformed[col] = np.log(s).diff()
        transform_type[col] = "logdiff"
    else:
        transformed[col] = s.pct_change()
        transform_type[col] = "pct_change"

transformed = transformed.replace([np.inf, -np.inf], np.nan)

# ----------------------------
# 5) Transformed ADF
# ----------------------------
tr_rows = []
for col in num_cols:
    r = run_adf(transformed[col], ADF_AUTOLAG)
    tr_rows.append({"Variable": col, "Transform": transform_type[col], **r})

tr_adf = pd.DataFrame(tr_rows).sort_values(["status", "pvalue"], na_position="last")
tr_adf.to_csv("./adf_transformed.csv", index=False)
print("Saved: ./adf_transformed.csv")

# ----------------------------
# 6) VAR data
# ----------------------------
var_data = transformed.drop(columns=["Date"]).dropna()
print("Observations after transform & dropna:", len(var_data))

if len(var_data) <= (MAX_LAGS + 5):
    raise ValueError(f"Not enough observations for VAR with MAX_LAGS={MAX_LAGS}. "
                     f"Need > {MAX_LAGS+5}, got {len(var_data)}")

# ----------------------------
# 7) Lag selection score table (per lag)
# ----------------------------
model = VAR(var_data)

scores = []
for p in range(1, MAX_LAGS + 1):
    try:
        fit = model.fit(p)
        scores.append({
            "lag": p,
            "aic": fit.aic,
            "bic": fit.bic,
            "hqic": fit.hqic,
            "fpe": fit.fpe
        })
    except Exception as e:
        scores.append({
            "lag": p,
            "aic": np.nan,
            "bic": np.nan,
            "hqic": np.nan,
            "fpe": np.nan,
            "error": str(e)
        })

lag_scores = pd.DataFrame(scores)
lag_scores.to_csv("./lag_selection_scores.csv", index=False)
print("Saved: ./lag_selection_scores.csv")

# choose lag by minimum AIC among valid rows
valid = lag_scores.dropna(subset=["aic"])
if len(valid) == 0:
    raise ValueError("All lag fits failed; check data/collinearity/constant columns.")

p_aic = int(valid.loc[valid["aic"].idxmin(), "lag"])
p_bic = int(valid.loc[valid["bic"].idxmin(), "lag"]) if valid["bic"].notna().any() else np.nan
p_hqic = int(valid.loc[valid["hqic"].idxmin(), "lag"]) if valid["hqic"].notna().any() else np.nan
p_fpe = int(valid.loc[valid["fpe"].idxmin(), "lag"]) if valid["fpe"].notna().any() else np.nan

choice = pd.DataFrame([{
    "selected_by": "min_value",
    "p_aic": p_aic,
    "p_bic": p_bic,
    "p_hqic": p_hqic,
    "p_fpe": p_fpe,
    "max_lags": MAX_LAGS,
    "nobs_used": len(var_data),
    "n_vars": var_data.shape[1]
}])
choice.to_csv("./lag_selection_choice.csv", index=False)
print("Saved: ./lag_selection_choice.csv")

# ----------------------------
# 8) Fit VAR with AIC-selected lag and save params/cov
# ----------------------------
var_model = model.fit(p_aic)

var_model.params.to_csv("./var_params.csv")
print("Saved: ./var_params.csv")

cov_df = pd.DataFrame(var_model.sigma_u, index=var_data.columns, columns=var_data.columns)
cov_df.to_csv("./var_residual_cov.csv")
print("Saved: ./var_residual_cov.csv")

print("\nDone.")
print("Selected lag (AIC):", p_aic)

Shape: (1325, 11)
Date range: 2020-10-13 00:00:00 ~ 2026-01-12 00:00:00
Columns: ['SOLVPN', 'COPPER', 'DXY', 'UST10Y', 'VIX', 'dlog_SOLVPN', 'dlog_COPPER', 'dlog_DXY', 'd_UST10Y', 'dlog_VIX']
Saved: ./adf_raw.csv
Saved: ./adf_transformed.csv
Observations after transform & dropna: 1287


c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)


Saved: ./lag_selection_scores.csv
Saved: ./lag_selection_choice.csv
Saved: ./var_params.csv
Saved: ./var_residual_cov.csv

Done.
Selected lag (AIC): 1
